# 中国船舶 (600150.SH) 技术指标计算 Notebook

**数据来源**: Tushare Pro (via MCP) | **时间范围**: 2025-07-09 ~ 2026-07-08  | **交易日数**: 约230天

本 Notebook 逐步骤演示 RSI、MACD、布林带、ATR 四个经典技术指标的计算过程与可视化。

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import rcParams

# 中文字体设置
rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False

# 加载已从 Tushare 获取的日线数据
with open("data/中国船舶_A.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# 构建 DataFrame，按日期升序
df = pd.DataFrame(raw)
df['trade_date'] = pd.to_datetime(df['trade_date'])
df = df.sort_values('trade_date').reset_index(drop=True)
df = df.rename(columns={'vol': 'volume', 'amount': 'amount_k'})

print(f"数据概览: {len(df)} 个交易日")
print(f"日期范围: {df['trade_date'].min().date()} ~ {df['trade_date'].max().date()}")
print(f"最新收盘: {df['close'].iloc[-1]:.2f}  最低: {df['close'].min():.2f}  最高: {df['close'].max():.2f}")
df.head(8)


In [ ]:
# 快速可视化：收盘价 + 成交量
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df['trade_date'], df['close'], color='#1a1a2e', linewidth=1.2)
ax1.set_ylabel('收盘价 (元)', fontsize=12)
ax1.set_title('中国船舶 600150.SH  日线收盘价', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

colors = ['#e74c3c' if v >= 0 else '#27ae60' for v in df['pct_chg']]
ax2.bar(df['trade_date'], df['volume']/10000, color=colors, width=0.8)
ax2.set_ylabel('成交量 (万手)', fontsize=12)
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()


## §1 RSI — 相对强弱指标 (14日)

**原理**: RSI 衡量近期涨幅与跌幅的相对强弱。值域 [0, 100]，>70 超买，<30 超卖。

**计算步骤**:
1. 逐日计算涨幅 U = max(close - pre_close, 0)，跌幅 D = max(pre_close - close, 0)
2. 用 Wilder 指数平滑递推 (alpha = 1/14)
3. RS = AvgU / AvgD  →  RSI = 100 - 100/(1+RS)

In [ ]:
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)

df['rsi_14'] = calc_rsi(df['close'], 14)

# 检查值域
print(f"RSI 范围: [{df['rsi_14'].min():.2f}, {df['rsi_14'].max():.2f}]")
print(f"最近 5 日 RSI:\n{df[['trade_date', 'close', 'rsi_14']].tail()}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1.5]})

ax1.plot(df['trade_date'], df['close'], color='#1a1a2e', linewidth=1.2)
ax1.set_ylabel('收盘价', fontsize=12)
ax1.set_title('§1 RSI (14) — 中国船舶 600150.SH', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

ax2.plot(df['trade_date'], df['rsi_14'], color='#534AB7', linewidth=1.2)
ax2.axhline(y=70, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.7, label='超买 70')
ax2.axhline(y=30, color='#27ae60', linestyle='--', linewidth=0.8, alpha=0.7, label='超卖 30')
ax2.axhline(y=50, color='#888780', linestyle=':', linewidth=0.5, alpha=0.5, label='中性 50')
ax2.fill_between(df['trade_date'], 70, df['rsi_14'], where=df['rsi_14']>70, color='#e74c3c', alpha=0.1)
ax2.fill_between(df['trade_date'], 30, df['rsi_14'], where=df['rsi_14']<30, color='#27ae60', alpha=0.1)
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI', fontsize=12)
ax2.legend(fontsize=10, loc='upper left')
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()


## §2 MACD — 异同移动平均线 (12, 26, 9)

**原理**: 快 EMA(12) - 慢 EMA(26) = DIF，DIF 的 EMA(9) = DEA(信号线)，柱状图 BAR = 2×(DIF-DEA)

**信号**: DIF 上穿 DEA → 金叉 (买入)；DIF 下穿 DEA → 死叉 (卖出)

In [ ]:
def calc_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    dif = ema_fast - ema_slow
    dea = dif.ewm(span=signal, adjust=False).mean()
    bar = 2 * (dif - dea)
    return dif, dea, bar

df['dif'], df['dea'], df['bar'] = calc_macd(df['close'])

# 识别金叉/死叉
df['cross'] = 0
for i in range(1, len(df)):
    if df['dif'].iloc[i-1] <= df['dea'].iloc[i-1] and df['dif'].iloc[i] > df['dea'].iloc[i]:
        df.loc[df.index[i], 'cross'] = 1   # 金叉
    elif df['dif'].iloc[i-1] >= df['dea'].iloc[i-1] and df['dif'].iloc[i] < df['dea'].iloc[i]:
        df.loc[df.index[i], 'cross'] = -1  # 死叉

golden = df[df['cross'] == 1]
death  = df[df['cross'] == -1]
print(f"金叉次数: {len(golden)}  死叉次数: {len(death)}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1.5]})

ax1.plot(df['trade_date'], df['close'], color='#1a1a2e', linewidth=1.2)
ax1.set_ylabel('收盘价', fontsize=12)
ax1.set_title('§2 MACD (12, 26, 9) — 中国船舶 600150.SH', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

ax2.plot(df['trade_date'], df['dif'], color='#378ADD', linewidth=1.2, label='DIF')
ax2.plot(df['trade_date'], df['dea'], color='#EF9F27', linewidth=1.0, linestyle='--', label='DEA')
colors_bar = ['#e74c3c' if v >= 0 else '#27ae60' for v in df['bar']]
ax2.bar(df['trade_date'], df['bar'], color=colors_bar, width=0.8, alpha=0.7)
ax2.axhline(y=0, color='#888780', linewidth=0.5)
# 标注金叉死叉
ax2.scatter(golden['trade_date'], golden['dif'], color='#e74c3c', marker='^', s=60, zorder=5, label='金叉')
ax2.scatter(death['trade_date'], death['dif'], color='#27ae60', marker='v', s=60, zorder=5, label='死叉')
ax2.legend(fontsize=9, ncol=5, loc='upper left')
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()


## §3 布林带 Bollinger Bands (20, 2)

**原理**: 中轨 = SMA(20)，上下轨 = 中轨 ± 2×标准差。约95%的价格落在通道内。

**信号**:
- Band Squeeze: 带宽收窄 → 变盘预警
- Walking the Bands: 价格持续贴边 → 趋势强劲

In [ ]:
def calc_bollinger(series, period=20, k=2):
    mid = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    upper = mid + k * std
    lower = mid - k * std
    return mid, upper, lower

df['boll_mid'], df['boll_upper'], df['boll_lower'] = calc_bollinger(df['close'])

# 计算 %B 和带宽
df['boll_width'] = df['boll_upper'] - df['boll_lower']
df['boll_pct_b'] = (df['close'] - df['boll_lower']) / (df['boll_upper'] - df['boll_lower']) * 100

# 找出近3个月最窄的5个点（变盘预警）
recent = df[df['trade_date'] > '2026-04-01']
squeeze = recent.nsmallest(5, 'boll_width')
print(f"带宽范围: [{df['boll_width'].min():.2f}, {df['boll_width'].max():.2f}]")
print(f"当前带宽: {df['boll_width'].iloc[-1]:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

ax.fill_between(df['trade_date'], df['boll_upper'], df['boll_lower'], color='#B5D4F4', alpha=0.15)
ax.plot(df['trade_date'], df['boll_upper'], color='#378ADD', linewidth=0.6, alpha=0.4)
ax.plot(df['trade_date'], df['boll_mid'], color='#888780', linewidth=0.8, linestyle='--', alpha=0.5)
ax.plot(df['trade_date'], df['boll_lower'], color='#378ADD', linewidth=0.6, alpha=0.4)
ax.plot(df['trade_date'], df['close'], color='#1a1a2e', linewidth=1.5)
ax.fill_between(df['trade_date'], df['boll_upper'], df['close'], where=df['close']>df['boll_upper'], color='#e74c3c', alpha=0.08)
ax.fill_between(df['trade_date'], df['boll_lower'], df['close'], where=df['close']<df['boll_lower'], color='#27ae60', alpha=0.08)

# 标注 squeeze 点
ax.scatter(squeeze['trade_date'], squeeze['close'], color='#EF9F27', marker='o', s=50, zorder=5)
for _, r in squeeze.iterrows():
    ax.annotate('收窄', (r['trade_date'], r['close']), textcoords="offset points", xytext=(0,12), fontsize=8, color='#EF9F27', ha='center')

ax.set_title('§3 Bollinger Bands (20, 2) — 中国船舶 600150.SH', fontsize=14, fontweight='bold')
ax.set_ylabel('价格 (元)', fontsize=12)
ax.legend(['上轨','中轨 SMA20','下轨','收盘价'], fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()


## §4 ATR — 平均真实波幅 (14日)

**原理**: 真实波幅 TR = max(当日高低差, |高-昨收|, |低-昨收|)，再指数平滑。

**用途**: 动态止损 (2×ATR)、仓位管理 (风险金额÷ATR)、波动率过滤

In [ ]:
def calc_atr(df, period=14):
    high, low, close = df['high'], df['low'], df['close']
    prev_close = close.shift(1)
    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1/period, adjust=False).mean()
    return tr, atr

df['tr'], df['atr_14'] = calc_atr(df)

print(f"TR 范围: [{df['tr'].min():.2f}, {df['tr'].max():.2f}]")
print(f"ATR 范围: [{df['atr_14'].min():.2f}, {df['atr_14'].max():.2f}]")
print(f"当前 ATR(14): {df['atr_14'].iloc[-1]:.2f}")
print(f"\n基于 ATR 的止损建议 (当前收盘 {df['close'].iloc[-1]:.2f}):")
print(f"  1x ATR 止损: {df['close'].iloc[-1] - df['atr_14'].iloc[-1]:.2f}")
print(f"  2x ATR 止损: {df['close'].iloc[-1] - 2*df['atr_14'].iloc[-1]:.2f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True, gridspec_kw={'height_ratios': [2.5, 1.5]})

ax1.plot(df['trade_date'], df['close'], color='#1a1a2e', linewidth=1.2)
ax1.set_ylabel('收盘价', fontsize=12)
ax1.set_title('§4 ATR (14) — 中国船舶 600150.SH', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

# ATR + 动态止损带
atr_val = df['atr_14'].iloc[-1]
ax2.plot(df['trade_date'], df['atr_14'], color='#EF9F27', linewidth=1.2)
ax2.fill_between(df['trade_date'], 0, df['atr_14'], color='#FAC775', alpha=0.2)
ax2.set_ylabel('ATR (元)', fontsize=12)
ax2.axhline(y=atr_val, color='#BA7517', linestyle='--', linewidth=0.8, alpha=0.7, label=f'当前 ATR={atr_val:.2f}')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()


## §5 综合技术看板

将收盘价 + RSI + MACD + 布林带 + ATR 集成到一个多面板视图中，便于一次性查看所有指标信号。

In [ ]:
# 取近 120 个交易日，使图表清晰可读
df_viz = df.tail(120).copy()

fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(5, 1, height_ratios=[3, 1.5, 1.5, 1.5, 1.5], hspace=0.08)

# Panel 1: 收盘价 + 布林带
ax1 = fig.add_subplot(gs[0])
ax1.fill_between(df_viz['trade_date'], df_viz['boll_upper'], df_viz['boll_lower'], color='#B5D4F4', alpha=0.15)
ax1.plot(df_viz['trade_date'], df_viz['boll_upper'], color='#378ADD', linewidth=0.5, alpha=0.3)
ax1.plot(df_viz['trade_date'], df_viz['boll_mid'], color='#888780', linewidth=0.6, alpha=0.4)
ax1.plot(df_viz['trade_date'], df_viz['boll_lower'], color='#378ADD', linewidth=0.5, alpha=0.3)
ax1.plot(df_viz['trade_date'], df_viz['close'], color='#1a1a2e', linewidth=1.5)
ax1.set_ylabel('收盘价', fontsize=11)
ax1.set_title('中国船舶 600150.SH  综合技术看板', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.tick_params(labelbottom=False)

# Panel 2: RSI
ax2 = fig.add_subplot(gs[1])
ax2.plot(df_viz['trade_date'], df_viz['rsi_14'], color='#534AB7', linewidth=1.2)
ax2.axhline(y=70, color='#e74c3c', linestyle='--', linewidth=0.5, alpha=0.6)
ax2.axhline(y=30, color='#27ae60', linestyle='--', linewidth=0.5, alpha=0.6)
ax2.axhline(y=50, color='#888', linestyle=':', linewidth=0.4, alpha=0.3)
ax2.fill_between(df_viz['trade_date'], 70, 100, color='#e74c3c', alpha=0.04)
ax2.fill_between(df_viz['trade_date'], 0, 30, color='#27ae60', alpha=0.04)
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI', fontsize=10)
ax2.grid(alpha=0.3)
ax2.tick_params(labelbottom=False)

# Panel 3: MACD
ax3 = fig.add_subplot(gs[2])
ax3.plot(df_viz['trade_date'], df_viz['dif'], color='#378ADD', linewidth=1.0, label='DIF')
ax3.plot(df_viz['trade_date'], df_viz['dea'], color='#EF9F27', linewidth=0.8, label='DEA')
bar_colors = ['#e74c3c' if v>=0 else '#27ae60' for v in df_viz['bar']]
ax3.bar(df_viz['trade_date'], df_viz['bar'], color=bar_colors, width=0.8, alpha=0.6)
ax3.axhline(y=0, color='#888', linewidth=0.4)
ax3.set_ylabel('MACD', fontsize=10)
ax3.legend(fontsize=8, ncol=2)
ax3.grid(alpha=0.3)
ax3.tick_params(labelbottom=False)

# Panel 4: ATR
ax4 = fig.add_subplot(gs[3])
ax4.plot(df_viz['trade_date'], df_viz['atr_14'], color='#EF9F27', linewidth=1.2)
ax4.fill_between(df_viz['trade_date'], 0, df_viz['atr_14'], color='#FAC775', alpha=0.15)
ax4.set_ylabel('ATR', fontsize=10)
ax4.grid(alpha=0.3)
ax4.tick_params(labelbottom=False)

# Panel 5: 成交量
ax5 = fig.add_subplot(gs[4])
vol_colors = ['#e74c3c' if v>=0 else '#27ae60' for v in df_viz['pct_chg']]
ax5.bar(df_viz['trade_date'], df_viz['volume']/10000, color=vol_colors, width=0.8, alpha=0.7)
ax5.set_ylabel('成交量\n(万手)', fontsize=10)
ax5.grid(alpha=0.3)
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
for label in ax5.get_xticklabels():
    label.set_fontsize(9)

plt.tight_layout()
plt.subplots_adjust(hspace=0.08)
plt.show()


## 总结

本 Notebook 使用标准参数计算了四个经典技术指标：

| 指标 | 参数 | 当前值 | 信号解读 |
|:---|:---|:---|:---|
| RSI | 14日 | 见图中 | >70 超买 / <30 超卖 |
| MACD | 12,26,9 | 见图中 | 金叉/死叉 + 零轴位置 |
| Bollinger | 20,2 | 见图中 | 带宽收窄预警变盘 |
| ATR | 14日 | 见图中 | 用于动态止损和仓位管理 |

数据来源：Tushare Pro · 生成时间：2026-07-09 · 规范文件：`spec.yaml`